# 05 · Mapas de profundidad con Video Depth Anything Small (S8)

**Historia:** SP-02.7 (spike de profundidad monocular). **Kernel:** `colibri-vda` (otro torch y el repo de VDA; ver
`entornos/instalar.md`). Es el único notebook que usa ese entorno.

Procesa el video completo dos veces (original y volteado horizontalmente) y guarda los mapas en float16 en
`runs/profundidad/cache_vda/<corto>_{orig,flip}.npz`, ya alineados con el frame original. El notebook 06 los lee con
el backend `vda` de `00_comun`.

- **Licencia (D-17):** solo el modelo Small (Apache-2.0). Base y Large son CC-BY-NC y no se usan.
- **Salida:** disparidad relativa, mayor = más cerca (al revés que YOLO26-Depth).
- **Memoria:** prueba resoluciones de entrada de mayor a menor (448, 392, 336, 280) hasta que cabe en la GPU
  (6 GB en la laptop, sin xFormers). La usada queda en `info.json`.

En modo `artefactos` no recalcula los mapas que ya existan: los `.npz` pesan cientos de MB y no van en git.

In [ ]:
%run 00_comun.ipynb
VIDEO_LIST = ["i5bE"]          # cortos de datos_derivados/fuentes.csv; SP-02.9 añadirá otro video de cámara fija
VDA_REPO = P["modelos"] / "Video-Depth-Anything"
CACHE = RUNS / "profundidad" / "cache_vda"; CACHE.mkdir(parents=True, exist_ok=True)
VIDEOS = {c: str(video_path(FUENTES.set_index("corto").loc[c].video_id)) for c in VIDEO_LIST}
SIZES = tuple(int(s) for s in os.environ.get("VDA_SIZES", "448,392,336,280").split(","))
print(VIDEOS, "| sizes", SIZES)

In [ ]:
import gc, time
import torch
sys.path.insert(0, str(VDA_REPO)); _cwd = os.getcwd(); os.chdir(VDA_REPO)
from video_depth_anything.video_depth import VideoDepthAnything
os.chdir(_cwd)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# Portado de video-val/resultados/profundidad/siete_puntos/vda_inferir.py
def read_all(path, short):
    cap = cv2.VideoCapture(path); fps = cap.get(cv2.CAP_PROP_FPS); out = []
    while True:
        ok, f = cap.read()
        if not ok: break
        s = short / min(f.shape[:2])
        out.append(cv2.cvtColor(cv2.resize(f, None, fx=s, fy=s, interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB))
    cap.release()
    return np.stack(out), fps

# Portado de video-val/resultados/profundidad/siete_puntos/vda_inferir.py
def main():
    m = VideoDepthAnything(encoder="vits", features=64, out_channels=[48, 96, 192, 384])
    m.load_state_dict(torch.load(VDA_REPO / "checkpoints" / "video_depth_anything_vits.pth", map_location="cpu", weights_only=True), strict=True)
    m = m.to("cuda").eval()
    used = {}
    for name, path in VIDEOS.items():
        for tag in ("orig", "flip"):
            out = CACHE / f"{name}_{tag}.npz"
            if out.exists(): continue
            for size in SIZES:
                try:
                    t0 = time.time(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
                    frames, fps = read_all(path, size)
                    if tag == "flip": frames = frames[:, :, ::-1].copy()
                    d, _ = m.infer_video_depth(frames, fps, input_size=size, device="cuda", fp32=False)
                    break
                except (torch.OutOfMemoryError, MemoryError) as e:
                    print(name, tag, "sin memoria a", size, type(e).__name__, flush=True); frames = d = None; gc.collect(); torch.cuda.empty_cache()
            else:
                raise RuntimeError("no cabe a ninguna resolución")
            d = np.asarray(d, dtype=np.float32)
            if tag == "flip": d = d[:, :, ::-1]
            np.savez_compressed(out, maps=d.astype(np.float16))
            used[f"{name}_{tag}"] = size
            print(name, tag, d.shape, f"input {size}", f"{len(d) / (time.time() - t0):.1f} fps", f"pico GPU {torch.cuda.max_memory_allocated() / 1e9:.2f} GB", "rango", float(d.min()), float(d.max()), flush=True)
            frames = d = None; gc.collect(); torch.cuda.empty_cache()
    (CACHE / "info.json").write_text(json.dumps(dict(modelo="vda-small", input_size=used, salida="disparidad relativa, mayor = más cerca"), indent=2), encoding="utf-8")

In [ ]:
pendientes = [f"{n}_{t}" for n in VIDEOS for t in ("orig", "flip") if not (CACHE / f"{n}_{t}.npz").exists()]
if pendientes or MODO == "recalcular":
    if MODO == "recalcular":
        print("modo recalcular: borra a mano los .npz que quieras rehacer; los existentes no se sobrescriben")
    main()
else:
    print("mapas ya calculados:", sorted(p.name for p in CACHE.glob("*.npz")))

In [ ]:
info = json.loads((CACHE / "info.json").read_text(encoding="utf-8")) if (CACHE / "info.json").exists() else {}
print(info)
for n in VIDEOS:
    d = np.load(CACHE / f"{n}_orig.npz")["maps"]
    n_frames = int(FUENTES.set_index("corto").loc[n].frames)
    print(n, d.shape, d.dtype, "| frames esperados", n_frames, "✓" if len(d) == n_frames else "✗")

## Conclusión (S8)

Los mapas de VDA Small tienen consistencia temporal (el modelo procesa el video completo), cosa que YOLO26-Depth
cuadro a cuadro no da (D-13). Se usan **solo** para leer la profundidad relativa en los 7 puntos (notebook 06).